In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

departments = {
    "WATCO Bhubaneswar": "49",
    "Rural Water Supply and Sanitation": "47"
}

def scrape_all_pages(driver):
    all_data = []
    while True:
        # Wait for the table to appear
        WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CSS_SELECTOR, "table.list_table")))
        table = driver.find_element(By.CSS_SELECTOR, "table.list_table")
        rows = table.find_elements(By.XPATH, ".//tr[starts-with(@id,'informal_')]")
        for row in rows:
            tds = row.find_elements(By.TAG_NAME, "td")
            if len(tds) < 6:
                continue
            all_data.append({
                "S.No": tds[0].text.strip(),
                "e-Published Date": tds[1].text.strip(),
                "Closing Date": tds[2].text.strip(),
                "Opening Date": tds[3].text.strip(),
                "Title/Ref": tds[4].text.strip(),
                "Organisation Chain": tds[5].text.strip(),
            })

        # Check if there is a "next" link enabled
        try:
            next_link = driver.find_element(By.XPATH, "//a[@id='linkFwd' and not(contains(@class, 'disabled'))]")
            driver.execute_script("arguments[0].scrollIntoView(true);", next_link)
            time.sleep(1)
            next_link.click()
            time.sleep(2)  # Wait for page transition
        except Exception:
            # No next page
            break
    return all_data

driver = webdriver.Chrome()
wait = WebDriverWait(driver, 60)

for dept, value in departments.items():
    print(f"Processing department: {dept}")
    driver.get("https://tendersodisha.gov.in/nicgep/app?page=FrontEndAdvancedSearch&service=page")

    # Select Organisation from dropdown
    wait.until(EC.presence_of_element_located((By.ID, "OrganisationName")))
    Select(driver.find_element(By.ID, "OrganisationName")).select_by_value(value)

    # At this point: Wait for user to solve CAPTCHA and click Search manually!
    print(f"Please manually enter CAPTCHA and click 'Search' for '{dept}'.")
    input("After results are loaded, press Enter to continue scraping...")

    # Scrape all pages
    data = scrape_all_pages(driver)

    # Save to Excel
    filename = f"{dept.replace(' ', '_')}_tenders.xlsx"
    pd.DataFrame(data).to_excel(filename, index=False)
    print(f"Saved {filename} with {len(data)} tenders.")

driver.quit()


Processing department: WATCO Bhubaneswar
Please manually enter CAPTCHA and click 'Search' for 'WATCO Bhubaneswar'.


After results are loaded, press Enter to continue scraping... 


Saved WATCO_Bhubaneswar_tenders.xlsx with 39 tenders.
Processing department: Rural Water Supply and Sanitation
Please manually enter CAPTCHA and click 'Search' for 'Rural Water Supply and Sanitation'.


After results are loaded, press Enter to continue scraping... 


Saved Rural_Water_Supply_and_Sanitation_tenders.xlsx with 7 tenders.


In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

departments = {
    "WATCO Bhubaneswar": "49",
    "Rural Water Supply and Sanitation": "47"
}

all_data = []
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 60)

for dept, value in departments.items():
    print(f"\nProcessing: {dept}")
    driver.get("https://tendersodisha.gov.in/nicgep/app?page=FrontEndAdvancedSearch&service=page")
    
    # Select 'Open Tender'
    wait.until(EC.presence_of_element_located((By.ID, "TenderType")))
    Select(driver.find_element(By.ID, "TenderType")).select_by_value("1")
    # Select department
    Select(driver.find_element(By.ID, "OrganisationName")).select_by_value(value)

    print("Please manually solve the CAPTCHA and click 'Search' for department:", dept)
    input("After results page loads, press Enter here to start scraping...")

    while True:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table.list_table")))
        table = driver.find_element(By.CSS_SELECTOR, "table.list_table")
        rows = table.find_elements(By.XPATH, ".//tr[starts-with(@id,'informal_')]")
        for row in rows:
            tds = row.find_elements(By.TAG_NAME, "td")
            if len(tds) < 6:
                continue
            all_data.append({
                "Department": dept,
                "S.No": tds[0].text.strip(),
                "e-Published Date": tds[1].text.strip(),
                "Closing Date": tds[2].text.strip(),
                "Opening Date": tds[3].text.strip(),
                "Title/Ref": tds[4].text.strip(),
                "Organisation Chain": tds[5].text.strip(),
            })
        # Try next page, else break
        try:
            next_link = driver.find_element(By.XPATH, "//a[@id='linkFwd' and not(contains(@class, 'disabled'))]")
            driver.execute_script("arguments[0].scrollIntoView(true);", next_link)
            time.sleep(1)
            next_link.click()
            time.sleep(2)
        except:
            break

# Save everything in one sheet
df = pd.DataFrame(all_data)
df.to_excel("tenders_all_departments.xlsx", index=False)
print(f"\nScraped {len(df)} tenders. Saved to 'tenders_all_departments.xlsx'.")

driver.quit()



Processing: WATCO Bhubaneswar
Please manually solve the CAPTCHA and click 'Search' for department: WATCO Bhubaneswar


After results page loads, press Enter here to start scraping... 



Processing: Rural Water Supply and Sanitation
Please manually solve the CAPTCHA and click 'Search' for department: Rural Water Supply and Sanitation


After results page loads, press Enter here to start scraping... 



Scraped 46 tenders. Saved to 'tenders_all_departments.xlsx'.
